In [1]:
import pandas as pd
import numpy as np
import time
from datetime import date
import pickle # page view data is in pickle format
import langcodes # for two-letter ISO 639 language codes

In [2]:
fileobject = open("Data/Species data/pageview_mammal_monthly.pkl","rb")
dct = pickle.load(fileobject)
fileobject.close()

Species: ID, Name

Languages: ID, Code, Name

Time: ID, Timestamp

Pageviews: ID, Time_ID, Species_ID, Language_ID, Views

In [3]:
## SPECIES - creating table
DF_SPECIES = pd.DataFrame(list(dct.keys()), columns = ['Latin_name'])
DF_SPECIES['ID'] = DF_SPECIES.index
DF_SPECIES

,Latin_name,ID
0,Coryphomys musseri,0
1,Rattus rattus,1
2,Lagothrix lagotricha,2
3,Petauroides ayamaruensis,3
4,Ochotona coreana,4
...,...,...
5767,Pipistrellus pulveratus,5767
5768,Plecotus christii,5768
5769,Proechimys trinitatis,5769
5770,Pygeretmus zhitkovi,5770


In [14]:
## TIME - creating table
# ID is actually timestamp string - is this ok? it's unique
DF_TIME_Times = list()
DF_TIME_Times_unformatted = list()

for year_id in range(0, 11): # hardcoded to be 2015-2025 right now, can be altered
    year = str(2015+year_id)
    for month_id in range(1,13):
        month = f"{month_id:02}"
        day = '01' # data could be granular to the day, in this case adjust here
        hour = '00' # always 00, midnight
        time_str = year+month+day+hour
        time = pd.to_datetime(time_str, format = "%Y%m%d%H") # example: 2015070100

        DF_TIME_Times_unformatted.append(time_str)
        DF_TIME_Times.append(time)

DF_TIME = pd.DataFrame(DF_TIME_Times, DF_TIME_Times_unformatted, columns = ['Datetime'])
DF_TIME['ID'] = DF_TIME.index.astype(int)
DF_TIME

,Datetime,ID
2015010100,2015-01-01,2015010100
2015020100,2015-02-01,2015020100
2015030100,2015-03-01,2015030100
2015040100,2015-04-01,2015040100
2015050100,2015-05-01,2015050100
...,...,...
2025080100,2025-08-01,2025080100
2025090100,2025-09-01,2025090100
2025100100,2025-10-01,2025100100
2025110100,2025-11-01,2025110100


In [15]:
## LANGUAGE - creating table
DF_LANGUAGES_Codes = list()

for spec in DF_SPECIES['Latin_name']: # string of species name
    languages = list(dct[spec].keys())
    DF_LANGUAGES_Codes.extend(languages)
    # Opportunity here to look at distribution of pages across languages

DF_LANGUAGES_Codes_set = set(DF_LANGUAGES_Codes)
DF_LANGUAGES = pd.DataFrame(DF_LANGUAGES_Codes_set, columns = ['Code'])
DF_LANGUAGES['ID'] = DF_LANGUAGES.index
DF_LANGUAGES['Name'] = [langcodes.Language.make(language = lang).display_name() for lang in DF_LANGUAGES['Code']]
DF_LANGUAGES

,Code,ID,Name
0,tw,0,Twi
1,arc,1,Aramaic
2,hr,2,Croatian
3,frp,3,Arpitan
4,bn,4,Bangla
...,...,...,...
312,ky,312,Kyrgyz
313,pnb,313,Western Panjabi
314,so,314,Somali
315,sq,315,Albanian


In [17]:
## PAGEVIEWS - creating table
species = list()
timestamps = list()
pageviews = list()
languages = list()

for spec in DF_SPECIES['Latin_name']:
    languages_spec = list(dct[spec].keys())
    spec_id = DF_SPECIES.loc[DF_SPECIES['Latin_name'] == spec]['ID'].item()
   
    for lang in languages_spec:
        timestamps_lang = dct[spec][lang]['timestamp']
        pageviews_lang = dct[spec][lang]['views']
        timestamps.extend(timestamps_lang)
        pageviews.extend(pageviews_lang)
        lang_id = DF_LANGUAGES.loc[DF_LANGUAGES['Code'] == lang]['ID'].item()
        languages.extend([lang_id]*len(timestamps_lang))
        species.extend([spec_id]*len(timestamps_lang))

In [22]:
DF_PAGEVIEWS = pd.DataFrame({'Timestamp_ID':timestamps, 'Species_ID':species, 'Language_ID':languages, 'Number_of_Pageviews':pageviews})
DF_PAGEVIEWS['ID'] = DF_PAGEVIEWS.index
DF_PAGEVIEWS['Timestamp_ID'] = DF_PAGEVIEWS['Timestamp_ID'].astype(int)
DF_PAGEVIEWS.dtypes
DF_PAGEVIEWS

,Timestamp_ID,Species_ID,Language_ID,Number_of_Pageviews,ID
0,2020110100,0,75,150,0
1,2020120100,0,75,92,1
2,2021010100,0,75,112,2
3,2021020100,0,75,86,3
4,2021030100,0,75,91,4
...,...,...,...,...,...
13730911,2025080100,5771,306,4,13730911
13730912,2025090100,5771,306,9,13730912
13730913,2025100100,5771,306,10,13730913
13730914,2025110100,5771,306,7,13730914


In [24]:
## Saving tables as pickle files
DF_SPECIES.to_pickle('./DF_SPECIES.pkl')
DF_LANGUAGES.drop('Code', axis=1) # 'Code' not in the database setup
DF_LANGUAGES.to_pickle('./DF_LANGUAGES.pkl')
DF_TIME.to_pickle('./DF_TIME.pkl')
DF_PAGEVIEWS.to_pickle('./DF_PAGEVIEWS.pkl')